Load all data

In [22]:
import pandas as pd
import numpy as np

BASE = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk/data"

cities = {
    'MY1':  'London',
    'BIRR': 'Birmingham',
    'MAN3': 'Manchester',
    'NEWC': 'Newcastle',
    'CARD': 'Cardiff',
}

dfs = []
for site_id, name in cities.items():
    df_train = pd.read_csv(f"{BASE}/{name}/merged/{site_id}_merged.csv",
                           parse_dates=['datetime'])
    df_test  = pd.read_csv(f"{BASE}/test/merged/{site_id}_merged_2025.csv",
                           parse_dates=['datetime'])
    dfs.append(df_train)
    dfs.append(df_test)

data = pd.concat(dfs, ignore_index=True)
data = data.sort_values(['city','datetime']).reset_index(drop=True)
data = data.drop_duplicates(subset=['city','datetime'])

print(f"Total rows: {len(data):,}")
print(f"Date range: {data['datetime'].min()} → {data['datetime'].max()}")
print(data.groupby('city').size())

Total rows: 219,120
Date range: 2021-01-01 01:00:00 → 2026-01-01 00:00:00
city
BIRR    43824
CARD    43824
MAN3    43824
MY1     43824
NEWC    43824
dtype: int64


Clip negatives and impute

In [23]:
# Clip negatives
for col in ['NO2','PM2.5','PM10','O3','SO2']:
    data[col] = data[col].clip(lower=0)

# Impute per city
def impute_city(df):
    df = df.copy().sort_values('datetime')
    cols = ['NO2','PM2.5','PM10','O3','temp','humidity','wind_speed','pressure']
    for col in cols:
        df[col] = df[col].ffill(limit=3)
        df[col] = df[col].fillna(df[col].median())
    df['SO2'] = df['SO2'].fillna(0)
    return df

data = data.groupby('city', group_keys=False).apply(impute_city)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

print("Missing after imputation:")
print(data[['NO2','PM2.5','PM10','O3','SO2','temp','humidity','wind_speed','pressure']].isna().sum())

Missing after imputation:
NO2           0
PM2.5         0
PM10          0
O3            0
SO2           0
temp          0
humidity      0
wind_speed    0
pressure      0
dtype: int64


C:\Users\nikes\AppData\Local\Temp\ipykernel_17156\3441894000.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(impute_city)


Build lag features per city

In [24]:
POLL_COLS = ['NO2','PM2.5','PM10','O3','SO2']
WTHR_COLS = ['temp','humidity','wind_speed','pressure']
ALL_COLS  = POLL_COLS + WTHR_COLS

LAG_HOURS    = [1, 2, 3, 6, 12, 24]
ROLLING_WINS = [6, 12, 24]

def build_lags(df):
    df = df.copy().sort_values('datetime').reset_index(drop=True)
    new_cols = {}

    for col in ALL_COLS:
        for lag in LAG_HOURS:
            new_cols[f'{col}_lag{lag}h'] = df[col].shift(lag)
        for win in ROLLING_WINS:
            new_cols[f'{col}_mean{win}h'] = df[col].shift(1).rolling(win).mean()
        for win in [6, 12]:
            new_cols[f'{col}_max{win}h'] = df[col].shift(1).rolling(win).max()

    # Build all new columns at once — no fragmentation
    lag_df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return lag_df

print("Building lag features per city...")
data = data.groupby('city', group_keys=False).apply(build_lags)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

lag_cols = [c for c in data.columns if any(x in c for x in ['lag','mean','max'])]
print(f"Lag feature columns created: {len(lag_cols)}")
print(f"Total columns now: {len(data.columns)}")

Building lag features per city...
Lag feature columns created: 99
Total columns now: 110


C:\Users\nikes\AppData\Local\Temp\ipykernel_17156\4142864821.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(build_lags)


Add time features

In [25]:
data['hour']        = data['datetime'].dt.hour
data['day_of_week'] = data['datetime'].dt.dayofweek
data['month']       = data['datetime'].dt.month
data['year']        = data['datetime'].dt.year
data['season']      = data['month'].apply(
    lambda m: 0 if m in [12,1,2] else 1 if m in [3,4,5] else 2 if m in [6,7,8] else 3
)
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)

city_map = {'MY1':0,'BIRR':1,'MAN3':2,'NEWC':3,'CARD':4}
data['city_code'] = data['city'].map(city_map)

print("Time features added.")
print(data[['datetime','hour','month','season','is_weekend','city_code']].head(3))

Time features added.
             datetime  hour  month  season  is_weekend  city_code
0 2021-01-01 01:00:00     1      1       0           0          1
1 2021-01-01 02:00:00     2      1       0           0          1
2 2021-01-01 03:00:00     3      1       0           0          1


Compute DAQI and shift target 24 hours forward

In [26]:
def compute_daqi(row):
    def sub_index(val, bands):
        if pd.isna(val): return 1
        for i, t in enumerate(bands):
            if val <= t: return i + 1
        return 10

    return max(
        sub_index(row['NO2'],   [67,134,200,267,334,400,467,534,600]),
        sub_index(row['PM2.5'], [11,23,35,41,47,53,58,64,70]),
        sub_index(row['PM10'],  [16,33,50,58,66,75,83,91,100]),
        sub_index(row['O3'],    [33,66,100,120,140,160,187,213,240]),
        sub_index(row['SO2'],   [88,177,266,354,443,532,710,887,1064]),
    )

print("Computing DAQI scores...")
data['DAQI_score'] = data.apply(compute_daqi, axis=1)

def daqi_band(s):
    if s <= 3: return 0
    if s <= 6: return 1
    if s <= 9: return 2
    return 3

data['DAQI_now'] = data['DAQI_score'].apply(daqi_band)

def shift_target(df):
    df = df.copy().sort_values('datetime')
    df['DAQI_6h'] = df['DAQI_now'].shift(-6)
    return df

data = data.groupby('city', group_keys=False).apply(shift_target)
data = data.sort_values(['city','datetime']).reset_index(drop=True)

# Drop NaN first, then merge classes
data = data.dropna(subset=['DAQI_6h'])
data['DAQI_6h'] = data['DAQI_6h'].astype(int)

# Merge High (2) and Very High (3) → Dangerous (2)
data['DAQI_6h'] = data['DAQI_6h'].replace({3: 2})

print(f"Rows after dropping no-target rows: {len(data):,}")
print(f"\n6h forecast target distribution (3-class):")
for i, label in enumerate(['Low', 'Moderate', 'Dangerous']):
    n = (data['DAQI_6h']==i).sum()
    print(f"  {i} {label:10s}: {n:6,}  ({n/len(data)*100:.1f}%)")

Computing DAQI scores...


C:\Users\nikes\AppData\Local\Temp\ipykernel_17156\2285979545.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('city', group_keys=False).apply(shift_target)


Rows after dropping no-target rows: 219,090

6h forecast target distribution (3-class):
  0 Low       : 214,401  (97.9%)
  1 Moderate  :  4,159  (1.9%)
  2 Dangerous :    530  (0.2%)


Drop rows with NaN lag features and define feature set

In [27]:
# First 24 rows per city have NaN lags — drop them
data = data.dropna(subset=lag_cols)
data = data.reset_index(drop=True)

print(f"Rows after dropping NaN lag rows: {len(data):,}")

# Define full feature set
TIME_FEATS = ['city_code','hour','day_of_week','month','season','is_weekend']
FEATURES   = TIME_FEATS + lag_cols

print(f"Total features: {len(FEATURES)}")
print(f"  Time features:  {len(TIME_FEATS)}")
print(f"  Lag features:   {len(lag_cols)}")

Rows after dropping NaN lag rows: 218,970
Total features: 105
  Time features:  6
  Lag features:   99


Train/test split and save

In [28]:
import os

train = data[data['year'] <= 2024].copy()
test  = data[data['year'] == 2025].copy()

print(f"Train: {len(train):,} rows  ({train['datetime'].min().date()} → {train['datetime'].max().date()})")
print(f"Test:  {len(test):,} rows   ({test['datetime'].min().date()} → {test['datetime'].max().date()})")

print(f"\nTrain target distribution:")
for i, label in enumerate(['Low','Moderate','High','Very High']):
    n = (train['DAQI_6h']==i).sum()
    print(f"  {i} {label:10s}: {n:6,}  ({n/len(train)*100:.1f}%)")

BASE_OUT = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk"
os.makedirs(f"{BASE_OUT}/models", exist_ok=True)

train.to_csv(f"{BASE_OUT}/models/train_forecast.csv", index=False)
test.to_csv(f"{BASE_OUT}/models/test_forecast.csv",  index=False)

# Save feature list so the app knows what to build
pd.Series(FEATURES).to_csv(f"{BASE_OUT}/models/feature_list.csv", index=False)

print(f"\nSaved:")
print(f"  models/train_forecast.csv — {len(train):,} rows")
print(f"  models/test_forecast.csv  — {len(test):,} rows")
print(f"  models/feature_list.csv   — {len(FEATURES)} features")

Train: 175,195 rows  (2021-01-02 → 2024-12-31)
Test:  43,775 rows   (2025-01-01 → 2025-12-31)

Train target distribution:
  0 Low       : 171,783  (98.1%)
  1 Moderate  :  3,007  (1.7%)
  2 High      :    405  (0.2%)
  3 Very High :      0  (0.0%)

Saved:
  models/train_forecast.csv — 175,195 rows
  models/test_forecast.csv  — 43,775 rows
  models/feature_list.csv   — 105 features
